# 03 — Stock-Level Feature Engineering

## AI-Based NIFTY 50 Portfolio Risk Prediction & Early Warning System

### Goal

Convert the clean historical price dataset into a **point-in-time stock feature dataset**.

Every feature must use only information available on or before the row's date.

### Input

`data/processed/clean_price_data.csv`

### Output

`data/processed/stock_features.csv`


## 1. Why are we creating stock-level features?

The raw dataset tells us what happened to each stock.

Our ML model ultimately needs signals that describe the stock's current condition:

- Is momentum improving or worsening?
- Is volatility increasing?
- Is the stock below its long-term trend?
- Is it experiencing a drawdown?
- Is trading volume unusually high?
- Is it becoming more sensitive to the market?

Later, these stock-level features will be aggregated into **portfolio-level risk features**.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

INPUT_PATH = Path("../data/processed/clean_price_data.csv")

df = pd.read_csv(INPUT_PATH, parse_dates=["Date"])
df = df.sort_values(["Ticker", "Date"]).reset_index(drop=True)

print("Input shape:", df.shape)
print("Tickers:", df["Ticker"].nunique())
print("Date range:", df["Date"].min(), "to", df["Date"].max())


## 2. Canonical daily return

We calculate the return ourselves:

\[
R_t = \frac{Close_t}{Close_{t-1}} - 1
\]

This keeps the feature-engineering pipeline independent of the original precomputed `Daily_Return` column.


In [ ]:
df["Return"] = df.groupby("Ticker")["Close"].pct_change()
df["Return_1D"] = df["Return"]

df[["Ticker", "Date", "Close", "Return"]].head(10)


## 3. Momentum features

We use several horizons:

- 5 trading days
- 10 trading days
- 20 trading days
- 60 trading days

These measure short-, medium-, and longer-term price momentum.


In [ ]:
for n in [5, 10, 20, 60]:
    df[f"Return_{n}D"] = df.groupby("Ticker")["Close"].pct_change(n)

df[[
    "Ticker", "Date", "Return_5D", "Return_10D",
    "Return_20D", "Return_60D"
]].tail()


## 4. Volatility features

Volatility is one of the most important ingredients in a portfolio-risk model.

We calculate rolling standard deviation of daily returns over:

- 5 days
- 20 days
- 60 days

We do **not** annualize these yet. For the ML model, consistency is more important at this stage.


In [ ]:
for window in [5, 20, 60]:
    df[f"Volatility_{window}D"] = (
        df.groupby("Ticker")["Return"]
          .transform(lambda s: s.rolling(window, min_periods=window).std())
    )

df[[
    "Ticker", "Date",
    "Volatility_5D", "Volatility_20D", "Volatility_60D"
]].tail()


## 5. Moving averages and trend features

Moving averages provide a simple representation of trend.

We calculate:

- MA 20
- MA 50
- MA 200

Then we create ratios such as:

\[
Close / MA_{200}
\]

A value below 1 means the stock is below its 200-day moving average.


In [ ]:
for window in [20, 50, 200]:
    df[f"MA_{window}D"] = (
        df.groupby("Ticker")["Close"]
          .transform(lambda s: s.rolling(window, min_periods=window).mean())
    )
    df[f"Price_to_MA_{window}D"] = df["Close"] / df[f"MA_{window}D"]

df["MA_50_to_MA_200"] = df["MA_50D"] / df["MA_200D"]

df[[
    "Ticker", "Date", "Close",
    "MA_20D", "MA_50D", "MA_200D",
    "Price_to_MA_200D", "MA_50_to_MA_200"
]].tail()


## 6. Drawdown features

Drawdown measures how far the current price is below a recent high.

For a trailing window:

\[
Drawdown = \frac{Current\ Price}{Trailing\ High} - 1
\]

We calculate 20-, 60-, and 200-day drawdowns.

These are particularly relevant because our final target is a **future portfolio drawdown event**.


In [ ]:
for window in [20, 60, 200]:
    rolling_high = (
        df.groupby("Ticker")["Close"]
          .transform(lambda s: s.rolling(window, min_periods=1).max())
    )
    df[f"Drawdown_{window}D"] = df["Close"] / rolling_high - 1

df[[
    "Ticker", "Date",
    "Drawdown_20D", "Drawdown_60D", "Drawdown_200D"
]].tail()


## 7. Volume and liquidity features

Unusual trading volume can accompany market stress.

We create:

- 20-day average volume
- volume ratio
- 1-day volume change

`Volume_Ratio_20D = 2` means today's volume is approximately twice its recent 20-day average.


In [ ]:
df["Volume_MA_20D"] = (
    df.groupby("Ticker")["Volume"]
      .transform(lambda s: s.rolling(20, min_periods=20).mean())
)

df["Volume_Ratio_20D"] = df["Volume"] / df["Volume_MA_20D"]
df["Volume_Change_1D"] = df.groupby("Ticker")["Volume"].pct_change()

df[[
    "Ticker", "Date", "Volume",
    "Volume_MA_20D", "Volume_Ratio_20D", "Volume_Change_1D"
]].tail()


## 8. Intraday range and ATR

The daily high-low range provides another view of short-term risk.

We calculate:

\[
Intraday\ Range = \frac{High-Low}{Close}
\]

We also calculate a 14-day average true range as a percentage of price.


In [ ]:
df["Prev_Close"] = df.groupby("Ticker")["Close"].shift(1)

tr = pd.concat([
    df["High"] - df["Low"],
    (df["High"] - df["Prev_Close"]).abs(),
    (df["Low"] - df["Prev_Close"]).abs()
], axis=1)

df["True_Range"] = tr.max(axis=1)

df["ATR_14D"] = (
    df.groupby("Ticker")["True_Range"]
      .transform(lambda s: s.rolling(14, min_periods=14).mean())
)

df["ATR_14D_Pct"] = df["ATR_14D"] / df["Close"]

df["Intraday_Range"] = (df["High"] - df["Low"]) / df["Close"]

df[[
    "Ticker", "Date",
    "Intraday_Range", "ATR_14D_Pct"
]].tail()


## 9. RSI

We calculate a standard 14-period RSI from historical daily returns.

Interpretation:

- Higher RSI → stronger recent momentum
- Lower RSI → weaker recent momentum

RSI is a secondary feature, not a standalone trading signal.


In [ ]:
def rsi(series, window=14):
    gain = series.clip(lower=0)
    loss = -series.clip(upper=0)

    avg_gain = gain.rolling(window, min_periods=window).mean()
    avg_loss = loss.rolling(window, min_periods=window).mean()

    rs = avg_gain / avg_loss.replace(0, np.nan)

    return 100 - (100 / (1 + rs))

df["RSI_14D"] = (
    df.groupby("Ticker")["Return"]
      .transform(lambda s: rsi(s, 14))
)

df[["Ticker", "Date", "RSI_14D"]].tail()


## 10. Market return proxy and rolling beta

We do not use the questionable static `Beta` column from the source dataset.

Instead, we construct a simple **equal-weight market return proxy** from the stocks available on each date.

Then:

\[
Beta = \frac{Cov(stock, market)}{Var(market)}
\]

using a rolling 60-trading-day window.

This is calculated from historical returns and therefore is point-in-time.


In [ ]:
market_daily_return = (
    df.groupby("Date")["Return"]
      .mean()
      .rename("Market_Return")
)

df = df.merge(
    market_daily_return,
    left_on="Date",
    right_index=True,
    how="left"
)

beta_values = pd.Series(index=df.index, dtype=float)

for ticker, group in df.groupby("Ticker", sort=False):
    cov = (
        group["Return"]
        .rolling(60, min_periods=60)
        .cov(group["Market_Return"])
    )

    var = (
        group["Market_Return"]
        .rolling(60, min_periods=60)
        .var()
    )

    beta_values.loc[group.index] = cov / var.replace(0, np.nan)

df["Rolling_Beta_60D"] = beta_values

df[[
    "Ticker", "Date",
    "Market_Return", "Rolling_Beta_60D"
]].tail()


## 11. Check feature missingness

Rolling features naturally have missing values at the beginning of each stock's history.

For example:

- MA 200 requires approximately 200 observations
- Volatility 60D requires 60 returns
- RSI 14D requires sufficient return history
- Rolling beta 60D requires 60 observations

**We will not fill these missing values with zero.**

Later, the portfolio/modeling stage will decide how much warm-up history to require.


In [ ]:
feature_missing = (
    df.isna().mean()
      .mul(100)
      .sort_values(ascending=False)
      .to_frame("Missing_Percent")
)

feature_missing.head(25)


## 12. Inspect feature distributions

We look at summary statistics for the generated numerical features.

The purpose is to identify:

- extreme values
- impossible values
- highly skewed features
- features that may require transformation later


In [ ]:
generated_features = [
    "Return_1D", "Return_5D", "Return_10D", "Return_20D", "Return_60D",
    "Volatility_5D", "Volatility_20D", "Volatility_60D",
    "Price_to_MA_20D", "Price_to_MA_50D", "Price_to_MA_200D",
    "MA_50_to_MA_200",
    "Drawdown_20D", "Drawdown_60D", "Drawdown_200D",
    "Volume_Ratio_20D", "Volume_Change_1D",
    "Intraday_Range", "ATR_14D_Pct", "RSI_14D",
    "Market_Return", "Rolling_Beta_60D"
]

df[generated_features].describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).T


## 13. Example stock feature visualization

We inspect one stock's:

- price
- 20-day volatility
- 200-day trend

This is exploratory only. It is not yet the portfolio model.


In [ ]:
example_ticker = df["Ticker"].dropna().iloc[0]
example = df[df["Ticker"] == example_ticker].copy()

plt.figure(figsize=(12, 5))
plt.plot(example["Date"], example["Close"], label="Close")
plt.plot(example["Date"], example["MA_200D"], label="MA 200D")
plt.title(f"{example_ticker}: Price and 200D Moving Average")
plt.xlabel("Date")
plt.ylabel("Price")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(example["Date"], example["Volatility_20D"])
plt.title(f"{example_ticker}: 20D Rolling Volatility")
plt.xlabel("Date")
plt.ylabel("Volatility")
plt.tight_layout()
plt.show()


## 14. Feature dictionary

The final project should document every feature rather than presenting an unexplained feature dump.

The complete dictionary is also saved to:

`reports/stock_feature_dictionary.csv`


In [ ]:
feature_dictionary = pd.DataFrame([
    ["Return_1D", "1-day percentage return", "Price"],
    ["Return_5D", "5-day percentage return", "Momentum"],
    ["Return_10D", "10-day percentage return", "Momentum"],
    ["Return_20D", "20-day percentage return", "Momentum"],
    ["Return_60D", "60-day percentage return", "Momentum"],
    ["Volatility_5D", "Rolling 5-day return standard deviation", "Risk"],
    ["Volatility_20D", "Rolling 20-day return standard deviation", "Risk"],
    ["Volatility_60D", "Rolling 60-day return standard deviation", "Risk"],
    ["MA_20D", "20-day moving average of Close", "Trend"],
    ["MA_50D", "50-day moving average of Close", "Trend"],
    ["MA_200D", "200-day moving average of Close", "Trend"],
    ["Price_to_MA_20D", "Close divided by 20-day MA", "Trend"],
    ["Price_to_MA_50D", "Close divided by 50-day MA", "Trend"],
    ["Price_to_MA_200D", "Close divided by 200-day MA", "Trend"],
    ["MA_50_to_MA_200", "50-day MA divided by 200-day MA", "Trend"],
    ["Drawdown_20D", "Current Close relative to trailing 20-day high", "Drawdown"],
    ["Drawdown_60D", "Current Close relative to trailing 60-day high", "Drawdown"],
    ["Drawdown_200D", "Current Close relative to trailing 200-day high", "Drawdown"],
    ["Volume_MA_20D", "20-day average volume", "Liquidity"],
    ["Volume_Ratio_20D", "Current volume divided by 20-day average", "Liquidity"],
    ["Volume_Change_1D", "1-day percentage change in volume", "Liquidity"],
    ["Intraday_Range", "(High-Low)/Close", "Risk"],
    ["ATR_14D_Pct", "14-day average true range divided by Close", "Risk"],
    ["RSI_14D", "14-day relative strength index", "Momentum"],
    ["Market_Return", "Equal-weight average stock return on the date", "Market"],
    ["Rolling_Beta_60D", "60-day rolling beta to dataset market proxy", "Market"],
], columns=["Feature", "Definition", "Category"])

feature_dictionary


## 15. Save the stock-level feature dataset

This dataset becomes the input for portfolio construction.

We keep the raw price columns because they may be useful later, while adding the engineered point-in-time features.


In [ ]:
feature_cols = [
    "Date", "Ticker", "Company_Name", "Sector",
    "Open", "High", "Low", "Close", "Volume",
    "Dividend", "Stock_Split",
    "Return_1D", "Return_5D", "Return_10D", "Return_20D", "Return_60D",
    "Volatility_5D", "Volatility_20D", "Volatility_60D",
    "MA_20D", "MA_50D", "MA_200D",
    "Price_to_MA_20D", "Price_to_MA_50D", "Price_to_MA_200D",
    "MA_50_to_MA_200",
    "Drawdown_20D", "Drawdown_60D", "Drawdown_200D",
    "Volume_MA_20D", "Volume_Ratio_20D", "Volume_Change_1D",
    "Intraday_Range", "ATR_14D_Pct", "RSI_14D",
    "Market_Return", "Rolling_Beta_60D",
]

stock_features = df[feature_cols].copy()

OUTPUT_PATH = Path("../data/processed/stock_features.csv")
stock_features.to_csv(OUTPUT_PATH, index=False)

DICT_PATH = Path("../reports/stock_feature_dictionary.csv")
feature_dictionary.to_csv(DICT_PATH, index=False)

print("Saved:", OUTPUT_PATH)
print("Shape:", stock_features.shape)


## 16. Final quality checks

Before using these features downstream, verify:

- one row per Date + Ticker
- dates are sorted
- no future-shifted feature was used
- no static/current fundamental field was introduced
- rolling features use historical/current observations only


In [ ]:
print("Duplicate Date + Ticker rows:",
      stock_features.duplicated(["Date", "Ticker"]).sum())

print("Sorted correctly:",
      stock_features.equals(
          stock_features.sort_values(["Ticker", "Date"]).reset_index(drop=True)
      ))

print("\nOutput columns:", len(stock_features.columns))
print("Output rows:", len(stock_features))


# Conclusion

We now have a **stock-level feature layer** built from the cleaned historical price data.

### Next stage

`04_portfolio_construction.ipynb`

will transform these individual stock observations into portfolio-level data.

We will initially use:

### Primary portfolio
**Equal-weight portfolio**

and then investigate whether alternative weighting schemes are useful.

The portfolio layer will calculate:

- portfolio return
- portfolio volatility
- portfolio drawdown
- stock concentration
- sector concentration
- cross-stock correlation

These portfolio-level variables will eventually become the inputs to the ML risk model.
